Установка библиотеки для парсинга тг-каналов

In [ ]:
pip install telethon

Выгрузка всех ключевых слов

In [ ]:
#Задаем функцию по вытаскиванию ключевых слов
def get_dictionary_word_list():
    # с помощью context manager мы гарантируем
    # файл будет закрыт при выходе из области видимости
    with open('СписокКлючевыхСлов.txt') as f:
        # возвращает split results, то есть все слова в файле
        lines = (line.rstrip() for line in f)
        lines = list(line for line in lines if line) # Непустые строки в списке
        return lines

Подключение аккаунта, по api которого идет подключение.

In [ ]:
from telethon.sync import TelegramClient, events
import asyncio

import re
import pandas as pd
import csv
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
from openpyxl.workbook.child import INVALID_TITLE_REGEX
from datetime import datetime

from telethon.tl.functions.messages import GetDialogsRequest
from telethon.tl.types import InputPeerEmpty, Channel, MessageMediaPhoto, MessageMediaDocument

api_id = 00000000 #id от аккаунта на который зарегестрирован через "Telegram - API Development Tools"
api_hash = 'dfkflgkflgfnsflkfo' #id hash от аккаунта на который зарегестрирован через "Telegram - API Development Tools"
phone = '89000000000' #номер телеграм-аккаунта под которым Telethon будет авторизовываться

client = TelegramClient(phone, api_id, api_hash)


**Проверка по всем тг-каналам, которые есть в подписках**

In [ ]:
DictionaryWord = get_dictionary_word_list()

print(DictionaryWord)

await client.start()
print("Клиент запущен!")

dialogs = await client.get_dialogs()
channels = [dialog for dialog in dialogs if dialog.is_channel]

print(f"У вас {len(channels)} каналов")

# Создание книги и листов
wb = Workbook()
ws = wb.active
ws.title = "Данные"

# Заголовки с форматированием
headers = ['Канал', 'Дата', 'Пост']
for col, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col, value=header)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal='center')

data = [[]]

date_of_post = datetime(2025, 10, 10)

for NewChannel in channels:

# Основной цикл для получения сообщений
# limit=None - означает, что мы хотим получить все сообщения
# limit=100 - получить последние 100 постов
#Получаем все посты начиная с указанной даты "date_of_post"
 async for message in client.iter_messages(NewChannel, reverse = True, offset_date = date_of_post):

       for KeyWord in DictionaryWord:
        if isinstance(message.text, str):
          #print("да это текст")
          if re.search(KeyWord, message.text, re.IGNORECASE):
           print(f'"{message.text}" содержит "{KeyWord}"')
           print(NewChannel.name)
           # Выводим дату и текст сообщения
           data.append([NewChannel.name, f"[{message.date}]", message.text or "<Сообщение без текста>"])
           break

print("\nГотово!")

for row_idx, row_data in enumerate(data, 2):
    for col_idx, value in enumerate(row_data, 1):
        ws.cell(row=row_idx, column=col_idx, value=value)

# Сохранение
wb.save('Выгрузка_постов.xlsx')
